# 06 — LLM Inference Walkthrough

## Position in pipeline
```
PDF -> Chunks -> Embeddings -> Retrieval -> Reranking -> **[06 LLM]** -> Answer
```

## Purpose

After reranking selects the top-k most relevant passages, the LLM generates a
natural-language answer grounded in those passages. This notebook walks through
every aspect of LLM inference in the project:

- **vLLM** serves the model (`Qwen/Qwen3-8B`) via an
  OpenAI-compatible API on `localhost:8000`.
- The `src/client/llm_client.py` module wraps the API call with a domain-specific
  system prompt and citation-aware user prompt.
- `enable_thinking=False` is passed via `extra_body` to suppress Qwen3's
  chain-of-thought `<think>...</think>` tokens for faster, cleaner RAG responses.
- Streaming mode enables token-by-token output for responsive UIs.
- Multi-turn conversation keeps a sliding window of prior turns.
- HyDE (Hypothetical Document Embeddings) expands the query before retrieval.

## Learning objectives

1. Inspect the system prompt and chat prompt template
2. Run non-streaming inference and examine the full response
3. Run streaming inference and observe incremental output
4. Simulate a multi-turn conversation with history
5. Use HyDE to generate a hypothetical passage for query expansion

## Imports

We import the LLM client module which provides `request_chat`, the prompt
templates, and the pre-configured OpenAI client pointing to vLLM. We also
import the HyDE client for query expansion.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import constant
from src.client.llm_client import (
    request_chat,
    llm_client,
    SYSTEM_PROMPT,
    LLM_CHAT_PROMPT,
)
from src.client.llm_hyde_client import request_hyde, LLM_HYDE_PROMPT

print(f"LLM model       : {constant.llm_model_name}")
print(f"vLLM endpoint   : {constant.vllm_base_url}")
print(f"Max tokens      : {constant.llm_max_tokens}")
print(f"Temperature     : {constant.llm_temperature}")
print(f"History window  : {constant.llm_history_window} turns")

## System prompt and chat prompt template

The system prompt establishes the LLM's role as an immunology expert that must
cite passages using `[N]` notation. The user prompt template injects the retrieved
context and the query into a structured format.

In [ ]:
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print()
print("=== CHAT PROMPT TEMPLATE ===")
print(LLM_CHAT_PROMPT)
print()

# Show what the actual prompt looks like once filled
example_context = (
    "[1] T cell activation requires recognition of peptide-MHC complexes "
    "by the T cell receptor (TCR), combined with co-stimulatory signals.\n\n"
    "[2] Without co-stimulation, T cells may become anergic rather than activated.\n\n"
    "[3] CD28 on T cells binds B7 molecules on APCs to provide the critical "
    "co-stimulatory signal for full T cell activation."
)
example_query = "What signals are required for T cell activation?"

filled_prompt = LLM_CHAT_PROMPT.format(context=example_context, query=example_query)
print("=== FILLED PROMPT (sent as user message) ===")
print(filled_prompt)

## Non-streaming inference

In non-streaming mode, `request_chat` waits for the full response before
returning. This is simpler to work with for batch evaluation but has higher
perceived latency in a UI.

**Prerequisite**: vLLM must be running at `localhost:8000`. If it is not available,
you will see a connection error.

In [ ]:
import time

query = "What signals are required for T cell activation?"
context = example_context  # reuse from above

print(f"Query: {query}\n")

t0 = time.perf_counter()
response = request_chat(query, context, stream=False)
elapsed = (time.perf_counter() - t0) * 1000

print(f"=== LLM Response (non-streaming, {elapsed:.0f} ms) ===")
print(response)
print(f"\nResponse length: {len(response)} characters")

## Streaming inference

Streaming returns an iterator of `ChatCompletionChunk` objects. Each chunk
contains a `delta.content` string (often a single token). This allows the UI
to display text incrementally as it is generated.

In [ ]:
print(f"Query: {query}\n")
print("=== LLM Response (streaming) ===")

t0 = time.perf_counter()
completion = request_chat(query, context, stream=True)

full_response = ""
token_count = 0
first_token_time = None

for chunk in completion:
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        full_response += delta
        token_count += 1
        print(delta, end="", flush=True)

total_time = (time.perf_counter() - t0) * 1000
ttft = (first_token_time - t0) * 1000 if first_token_time else 0

print(f"\n\n--- Streaming stats ---")
print(f"Time to first token (TTFT): {ttft:.0f} ms")
print(f"Total generation time:      {total_time:.0f} ms")
print(f"Tokens received:            {token_count}")
print(f"Tokens/sec:                 {token_count / (total_time / 1000):.1f}")

## Multi-turn conversation demo

The `chat_history` parameter lets us pass prior turns so the LLM can resolve
coreferences (e.g., "they" referring to T cells from a previous answer).
In the full pipeline, `RAGPipeline` manages this automatically with a sliding
window of `history_window` turns.

In [ ]:
# Turn 1
query_1 = "What are T helper cells?"
context_1 = (
    "[1] T helper cells (CD4+ T cells) are a subpopulation of T lymphocytes "
    "that coordinate immune responses by releasing cytokines.\n\n"
    "[2] They recognize antigens presented by MHC class II molecules on "
    "antigen-presenting cells."
)

print("=== Turn 1 ===")
print(f"User: {query_1}\n")
response_1 = request_chat(query_1, context_1, stream=False)
print(f"Assistant: {response_1}\n")

# Build history for Turn 2
chat_history = [
    {"role": "user", "content": query_1},
    {"role": "assistant", "content": response_1},
]

# Turn 2 — uses pronoun "they" referring to T helper cells
query_2 = "How are they activated?"
context_2 = (
    "[1] T helper cell activation requires two signals: TCR recognition of "
    "peptide-MHC class II, and co-stimulation via CD28-B7 interaction.\n\n"
    "[2] Activated T helper cells then differentiate into effector subsets "
    "(Th1, Th2, Th17, Treg) depending on the cytokine environment."
)

print("=== Turn 2 (with history) ===")
print(f"User: {query_2}\n")
response_2 = request_chat(query_2, context_2, stream=False, chat_history=chat_history)
print(f"Assistant: {response_2}\n")
print("Note: The LLM correctly interprets 'they' as T helper cells from Turn 1.")

## HyDE query expansion demo

**HyDE (Hypothetical Document Embeddings)** generates a short hypothetical
passage that would answer the query, then concatenates it with the original
query before retrieval. This bridges the vocabulary gap between question-style
queries and passage-style documents, improving recall.

The `request_hyde` function calls the same vLLM endpoint with a separate prompt
template that asks for a ~100-word hypothetical textbook passage.

In [ ]:
print("=== HyDE Prompt Template ===")
print(LLM_HYDE_PROMPT)
print()

query = "How do cytotoxic T lymphocytes kill target cells?"
print(f"Original query: {query}\n")

t0 = time.perf_counter()
hyde_text = request_hyde(query)
elapsed = (time.perf_counter() - t0) * 1000

print(f"=== HyDE expansion ({elapsed:.0f} ms) ===")
print(hyde_text)

expanded_query = f"{query}\n{hyde_text}"
print(f"\n=== Combined query for retrieval ===")
print(expanded_query)
print(f"\nOriginal length: {len(query)} chars")
print(f"Expanded length: {len(expanded_query)} chars")
print(f"\nThe expanded query contains domain-specific terminology that")
print(f"will better match relevant passages during dense retrieval.")

## Summary

### What this notebook demonstrated
- The system prompt and chat prompt template used for grounded QA
- Non-streaming inference: simple, full response at once
- Streaming inference: token-by-token output with TTFT and throughput metrics
- Multi-turn conversation using `chat_history` for coreference resolution
- HyDE query expansion for improved retrieval recall

### Key takeaways
- vLLM provides an OpenAI-compatible API, making it easy to swap models
- The system prompt enforces citation behavior (`[N]` notation) and prevents hallucination
- Streaming is essential for responsive UIs; TTFT is the key latency metric
- Multi-turn history is managed via a sliding window to stay within context limits
- HyDE is optional (controlled by `config.yaml: retrieval.hyde_enabled`)

### Next notebook
**`07_pipeline.ipynb`** — Full RAG pipeline end-to-end: query to answer with citations